# 🧭 ELEVIA COMPASS — Analyse & Modélisation

Ce notebook :
1. Charge les données brutes France Travail (API Offres d'emploi v2)
2. Construit les premières tables analytiques (fact_offers, dims, bridges)
3. Réalise une EDA de base (distributions, salaires, localisation)
4. Prépare la structure du graphe Compass (NetworkX)
5. Esquisse la structure pour Monte Carlo (matrice de transition)

Source des données : fichiers JSON présents dans `data/raw/`.

## 0. Configuration & Imports

- Définition des chemins
- Imports des librairies
- Fonctions utilitaires génériques (I/O, inspection)

In [ ]:
import json
import os
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from pprint import pprint

# Affichage pandas
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Chemins (adapter si besoin)
PROJECT_ROOT = Path(".").resolve()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

for d in [DATA_PROCESSED_DIR, DATA_INTERMEDIATE_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW_DIR:", DATA_RAW_DIR)

### Fonctions utilitaires génériques

- `load_json`
- `load_all_offers_json`
- helpers d'inspection

In [ ]:
def load_json(path: Path) -> Any:
    """Charge un fichier JSON (UTF-8) et renvoie l'objet Python."""
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def list_raw_offers_files(directory: Path) -> List[Path]:
    """Liste tous les fichiers d'offres FT dans data/raw."""
    return sorted(directory.glob("offres_*.json"))


def load_all_offers_json(directory: Path) -> List[Dict[str, Any]]:
    """
    Charge tous les fichiers d'offres et concatène les listes d'offres.
    Hypothèse : chaque fichier = dict avec clé 'resultats' OU une liste directe d'offres.
    """
    files = list_raw_offers_files(directory)
    all_offers = []
    print(f"➡️  {len(files)} fichiers d'offres trouvés dans {directory}")

    for path in files:
        data = load_json(path)
        if isinstance(data, dict) and "resultats" in data:
            offers = data["resultats"]
        elif isinstance(data, list):
            offers = data
        else:
            print(f"⚠️ Format inattendu pour {path.name}, ignoré.")
            continue

        all_offers.extend(offers)

    print(f"✅ Total d'offres chargées : {len(all_offers)}")
    return all_offers


def inspect_example_structure(objs: List[Dict[str, Any]], n: int = 3) -> None:
    """Affiche quelques exemples pour comprendre la structure brute."""
    for i, obj in enumerate(objs[:n]):
        print(f"\n=== Exemple offre {i} ===")
        pprint(obj)

## 1. Chargement & Inspection des données RAW

On charge :
- Tous les fichiers d'offres dans `data/raw/`
- On inspecte la structure brute pour vérifier les champs clés

In [ ]:
raw_offers = load_all_offers_json(DATA_RAW_DIR)

# Inspection rapide de la structure
inspect_example_structure(raw_offers, n=2)

## 2. Construction des DataFrames analytiques

Objectif : passer du JSON brut à des tables analytiques normalisées.

- `fact_offers` : chaque ligne = une offre
- `dim_location` : normalisation des lieux
- (Plus tard) `dim_job`, `dim_skill`, bridges...

Pour l'instant, on se concentre sur ce qu'on a : les OFFRES.

In [ ]:
def parse_location(lieu_travail: Optional[Dict[str, Any]]) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    """
    Extrait ville / département / région à partir du bloc 'lieuTravail'.
    Cette fonction est volontairement tolérante (clé manquante -> None).
    """
    if not isinstance(lieu_travail, dict):
        return None, None, None

    libelle = lieu_travail.get("libelle")  # ex: "78 - Saint-Germain-en-Laye"
    ville = lieu_travail.get("commune")
    code_postal = lieu_travail.get("codePostal")

    # Très basique : on sépare sur " - " si possible
    departement = None
    if isinstance(libelle, str) and " - " in libelle:
        departement = libelle.split(" - ", 1)[0].strip()

    return ville, departement, code_postal


def parse_salary(offer: Dict[str, Any]) -> Tuple[Optional[float], Optional[float]]:
    """
    Essaie d'extraire un salaire min / max si présent.
    La structure exacte dépend de l'API, donc on reste défensif.
    """
    salaire = offer.get("salaire")
    if not isinstance(salaire, dict):
        return None, None

    # Adapter à la structure réelle si différente
    min_val = salaire.get("borneMin")
    max_val = salaire.get("borneMax")

    try:
        min_val = float(min_val) if min_val is not None else None
    except (TypeError, ValueError):
        min_val = None

    try:
        max_val = float(max_val) if max_val is not None else None
    except (TypeError, ValueError):
        max_val = None

    return min_val, max_val


def extract_skills_from_offer(offer: Dict[str, Any]) -> List[str]:
    """
    Extrait une liste d'identifiants ou labels de compétences depuis l'offre, si présent.
    Placeholder : à adapter quand on aura la structure réelle des compétences.
    """
    # Exemple : offer.get("competences", [])
    competences = offer.get("competences", [])
    skills = []

    if isinstance(competences, list):
        for c in competences:
            if isinstance(c, dict):
                # Priorité à l'id, sinon au libellé
                skill_id = c.get("code") or c.get("libelle")
                if skill_id:
                    skills.append(str(skill_id).strip())
            elif isinstance(c, str):
                skills.append(c.strip())

    return skills


def build_fact_offers(raw_offers: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Construit la table fact_offers à partir des offres brutes.
    On extrait les champs stables / intéressants pour Compass.
    """
    records = []

    for o in raw_offers:
        offer_id = o.get("id")
        title = o.get("intitule")
        contract_type = o.get("typeContrat")
        description = o.get("description")
        date_creation = o.get("dateCreation")

        # Bloc entreprise
        entreprise = o.get("entreprise", {}) or {}
        company_name = entreprise.get("nom")
        company_description = entreprise.get("description")

        # Bloc lieuTravail
        lieu_travail = o.get("lieuTravail", {}) or {}
        city, dept, postal_code = parse_location(lieu_travail)

        # ROME / métier
        rome_code = o.get("romeCode") or o.get("romeCodeROME")
        rome_libelle = o.get("romeLibelle") or o.get("romeLibelleROME")

        # Salaire
        salary_min, salary_max = parse_salary(o)

        # Compétences (placeholder)
        skills_list = extract_skills_from_offer(o)

        record = {
            "offer_id": offer_id,
            "title": title,
            "company_name": company_name,
            "company_description": company_description,
            "contract_type": contract_type,
            "date_creation": pd.to_datetime(date_creation, errors="coerce"),
            "city": city,
            "department": dept,
            "postal_code": postal_code,
            "rome_code": rome_code,
            "rome_label": rome_libelle,
            "salary_min": salary_min,
            "salary_max": salary_max,
            "skills": skills_list,
        }
        records.append(record)

    df = pd.DataFrame.from_records(records)
    return df

In [ ]:
fact_offers = build_fact_offers(raw_offers)
print(f"✅ fact_offers shape: {fact_offers.shape}")
fact_offers.head()

### 2.1 Table dim_location (optionnelle mais utile)

On normalise les informations de localisation pour :
- faciliter les analyses par zone
- éventuellement enrichir avec lat/lon plus tard

In [ ]:
def build_dim_location(fact_offers: pd.DataFrame) -> pd.DataFrame:
    cols = ["city", "department", "postal_code"]
    dim = (
        fact_offers[cols]
        .drop_duplicates()
        .reset_index(drop=True)
        .assign(location_id=lambda df: df.index.astype(str))
    )
    return dim


dim_location = build_dim_location(fact_offers)
print(f"✅ dim_location shape: {dim_location.shape}")
dim_location.head()

### 2.2 Placeholders pour dim_job, dim_skill & bridges

Pour l'instant, on ne dispose pas encore des API ROME/Compétences/Marché (scopes manquants).
On prépare donc des fonctions "stub" pour avoir l'architecture prête.

In [ ]:
def build_dim_job_placeholder(fact_offers: pd.DataFrame) -> pd.DataFrame:
    """
    Placeholder : construit une dim_job basique à partir des rome_code / rome_label
    trouvés dans les offres. À remplacer par une dim_job propre quand ROME sera dispo.
    """
    dim_job = (
        fact_offers[["rome_code", "rome_label"]]
        .dropna(subset=["rome_code"])
        .drop_duplicates()
        .reset_index(drop=True)
    )
    dim_job = dim_job.rename(
        columns={
            "rome_code": "job_code",
            "rome_label": "job_label",
        }
    )
    return dim_job


def build_dim_skill_placeholder(fact_offers: pd.DataFrame) -> pd.DataFrame:
    """
    Placeholder : extrait les skills à partir des listes 'skills' présentes dans fact_offers.
    À remplacer par la vraie dim_skill basée sur l'API ROME competences.
    """
    all_skills = set()
    for skills in fact_offers["skills"]:
        if isinstance(skills, list):
            for s in skills:
                all_skills.add(s)

    dim_skill = pd.DataFrame(
        [{"skill_id": s, "skill_label": s, "skill_category": None} for s in sorted(all_skills)]
    )
    return dim_skill


def build_offer_skill_bridge(fact_offers: pd.DataFrame) -> pd.DataFrame:
    """
    Construit la table bridge offre-skill à partir de fact_offers.skills.
    """
    rows = []
    for _, row in fact_offers.iterrows():
        offer_id = row["offer_id"]
        skills = row["skills"]
        if not isinstance(skills, list):
            continue
        for s in skills:
            rows.append({"offer_id": offer_id, "skill_id": s})

    return pd.DataFrame(rows)

In [ ]:
dim_job = build_dim_job_placeholder(fact_offers)
dim_skill = build_dim_skill_placeholder(fact_offers)
bridge_offer_skill = build_offer_skill_bridge(fact_offers)

print("✅ dim_job:", dim_job.shape)
print("✅ dim_skill:", dim_skill.shape)
print("✅ bridge_offer_skill:", bridge_offer_skill.shape)

dim_job.head(), dim_skill.head(), bridge_offer_skill.head()

### 2.3 Sauvegarde des tables normalisées

On stocke les tables dans `data/processed/` pour réutilisation par d'autres scripts
(API backend, algorithmes de matching, etc.)

In [ ]:
def save_df(df: pd.DataFrame, name: str, directory: Path = DATA_PROCESSED_DIR) -> None:
    path_csv = directory / f"{name}.csv"
    path_parquet = directory / f"{name}.parquet"
    df.to_csv(path_csv, index=False)
    df.to_parquet(path_parquet, index=False)
    print(f"💾 Saved {name} → {path_csv.name}, {path_parquet.name}")


save_df(fact_offers, "fact_offers")
save_df(dim_location, "dim_location")
save_df(dim_job, "dim_job")
save_df(dim_skill, "dim_skill")
save_df(bridge_offer_skill, "bridge_offer_skill")

## 3. Analyses Exploratoires (EDA)

Objectifs :
- Comprendre la distribution des métiers, des lieux, des types de contrat
- Analyser les salaires (si disponibles)
- Avoir quelques visualisations de base

In [ ]:
def plot_bar_top_counts(
    df: pd.DataFrame,
    col: str,
    n: int = 20,
    title: Optional[str] = None,
    rotation: int = 45,
    figsize: Tuple[int, int] = (10, 5),
    save_name: Optional[str] = None,
):
    counts = df[col].value_counts().head(n)
    plt.figure(figsize=figsize)
    counts.plot(kind="bar")
    plt.title(title or f"Top {n} {col}")
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()

    if save_name:
        out_path = FIGURES_DIR / f"{save_name}.png"
        plt.savefig(out_path, dpi=150)
        print(f"💾 Figure saved: {out_path}")

    plt.show()


def plot_hist(
    df: pd.DataFrame,
    col: str,
    bins: int = 30,
    title: Optional[str] = None,
    figsize: Tuple[int, int] = (8, 4),
    save_name: Optional[str] = None,
):
    series = df[col].dropna()
    if series.empty:
        print(f"⚠️ No data to plot for {col}")
        return

    plt.figure(figsize=figsize)
    plt.hist(series, bins=bins)
    plt.title(title or f"Distribution de {col}")
    plt.tight_layout()

    if save_name:
        out_path = FIGURES_DIR / f"{save_name}.png"
        plt.savefig(out_path, dpi=150)
        print(f"💾 Figure saved: {out_path}")

    plt.show()

In [ ]:
# Top métiers (ROME) si disponibles
if "rome_code" in fact_offers.columns:
    plot_bar_top_counts(
        fact_offers,
        col="rome_code",
        n=20,
        title="Top 20 métiers (ROME) dans le corpus d'offres",
        save_name="top_rome_code",
    )

# Top villes
plot_bar_top_counts(
    fact_offers,
    col="city",
    n=20,
    title="Top 20 villes",
    save_name="top_cities",
)

# Types de contrat
plot_bar_top_counts(
    fact_offers,
    col="contract_type",
    n=10,
    title="Répartition des types de contrat",
    save_name="contract_type_distribution",
)

# Distribution des salaires min/max (si présents)
plot_hist(
    fact_offers,
    col="salary_min",
    title="Distribution des salaires minimum",
    save_name="salary_min_distribution",
)

plot_hist(
    fact_offers,
    col="salary_max",
    title="Distribution des salaires maximum",
    save_name="salary_max_distribution",
)

## 4. Construction du Graphe Compass (Version 0)

Graphe simplifié :
- Nœuds : jobs + skills
- Arêtes : job -- skill (à partir de `bridge_offer_skill` et `dim_job`/`dim_skill`)

Cette version utilise les placeholders. Elle sera enrichie
dès que les données ROME complètes seront disponibles.

In [ ]:
def build_compass_graph(dim_job: pd.DataFrame,
                        dim_skill: pd.DataFrame,
                        bridge_offer_skill: pd.DataFrame,
                        fact_offers: pd.DataFrame) -> nx.Graph:
    """
    Construit un graphe bipartite (jobs / skills).
    Version 0 : chaque (job_code, skill_id) est inféré à partir des offres.
    """
    G = nx.Graph()

    # Ajout des noeuds job
    for _, row in dim_job.iterrows():
        G.add_node(
            f"job::{row['job_code']}",
            node_type="job",
            job_code=row["job_code"],
            job_label=row["job_label"],
        )

    # Ajout des noeuds skill
    for _, row in dim_skill.iterrows():
        G.add_node(
            f"skill::{row['skill_id']}",
            node_type="skill",
            skill_id=row["skill_id"],
            skill_label=row["skill_label"],
        )

    # Bridge job-skill via les offres
    merged = (
        bridge_offer_skill.merge(
            fact_offers[["offer_id", "rome_code"]],
            on="offer_id",
            how="left",
        )
        .dropna(subset=["rome_code"])
    )

    # Création des arêtes job -- skill
    for _, row in merged.iterrows():
        job_node = f"job::{row['rome_code']}"
        skill_node = f"skill::{row['skill_id']}"
        if G.has_node(job_node) and G.has_node(skill_node):
            if G.has_edge(job_node, skill_node):
                # Incrémente un poids si déjà présent
                G[job_node][skill_node]["weight"] += 1
            else:
                G.add_edge(job_node, skill_node, weight=1)

    print(f"✅ Graphe Compass V0 : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes")
    return G

In [ ]:
G_compass = build_compass_graph(dim_job, dim_skill, bridge_offer_skill, fact_offers)

# Sauvegarde du graphe
graph_path = DATA_INTERMEDIATE_DIR / "G_compass_v0.gpickle"
nx.write_gpickle(G_compass, graph_path)
print(f"💾 Graphe sauvegardé → {graph_path}")

### 4.1 Métriques de base du graphe

- Degré des nœuds (job / skill)
- Centralité de degré
- Top hubs (jobs les plus connectés)

In [ ]:
def compute_graph_metrics(G: nx.Graph) -> pd.DataFrame:
    degrees = dict(G.degree())
    degree_centrality = nx.degree_centrality(G)

    records = []
    for node, deg in degrees.items():
        data = G.nodes[node]
        node_type = data.get("node_type")
        records.append(
            {
                "node": node,
                "node_type": node_type,
                "degree": deg,
                "degree_centrality": degree_centrality[node],
            }
        )

    df = pd.DataFrame(records)
    return df


graph_metrics = compute_graph_metrics(G_compass)
save_df(graph_metrics, "graph_metrics")

# Top jobs par degré
graph_metrics[graph_metrics["node_type"] == "job"].sort_values(
    "degree", ascending=False
).head(20)

## 5. Préparation Monte Carlo (Structure)

Pour l'instant, on ne dispose pas encore d'historique de transitions métiers,
ni de graphe métier-métier basé sur ROME.

On prépare néanmoins :
- la signature des fonctions
- le format de la matrice de transition

Ces fonctions seront remplies lorsque :
- les données ROME complètes seront disponibles
- les règles de similarité / transitions seront définies (Compass Engine).

In [ ]:
def build_job_transition_matrix(dim_job: pd.DataFrame,
                                G_compass: nx.Graph) -> pd.DataFrame:
    """
    Placeholder pour construire une matrice de transition job -> job.
    Version future :
    - similitude de compétences
    - proximité dans le graphe
    - patterns historiques
    """
    jobs = dim_job["job_code"].dropna().unique()
    jobs = sorted(jobs)

    # Matrice identité (transition vers soi-même) comme baseline
    M = pd.DataFrame(
        np.eye(len(jobs)),
        index=jobs,
        columns=jobs,
    )
    return M


def run_monte_carlo(
    start_job: str,
    transition_matrix: pd.DataFrame,
    steps: int = 10,
    simulations: int = 1000,
) -> pd.DataFrame:
    """
    Simule des trajectoires professionnelles en utilisant une matrice de transition.
    Version simple : tirage discret à chaque étape.
    """
    jobs = transition_matrix.index.to_list()
    job_to_idx = {j: i for i, j in enumerate(jobs)}

    if start_job not in job_to_idx:
        raise ValueError(f"start_job {start_job} absent de la matrice.")

    M = transition_matrix.values

    trajectories = []
    for sim in range(simulations):
        current_idx = job_to_idx[start_job]
        traj = [start_job]

        for _ in range(steps):
            probs = M[current_idx]
            # Normalisation défensive
            if probs.sum() == 0:
                probs = np.ones_like(probs) / len(probs)
            else:
                probs = probs / probs.sum()

            next_idx = np.random.choice(len(jobs), p=probs)
            next_job = jobs[next_idx]
            traj.append(next_job)
            current_idx = next_idx

        trajectories.append(traj)

    cols = [f"t{i}" for i in range(len(traj))]
    df_traj = pd.DataFrame(trajectories, columns=cols)
    return df_traj

In [ ]:
# Construction d'une matrice de transition baseline (identité)
transition_matrix = build_job_transition_matrix(dim_job, G_compass)
save_df(transition_matrix.reset_index().rename(columns={"index": "job_code"}), "transition_matrix_baseline")

# Exemple d'utilisation (si dim_job non vide)
if not dim_job.empty:
    example_job = dim_job["job_code"].iloc[0]
    traj_df = run_monte_carlo(
        start_job=example_job,
        transition_matrix=transition_matrix,
        steps=5,
        simulations=100,
    )
    print(f"Exemple de trajectoires à partir de {example_job}:")
    traj_df.head()

## 6. Récap & Prochaines étapes

✅ Ce notebook produit actuellement :
- `fact_offers`
- `dim_location`
- `dim_job` (placeholder)
- `dim_skill` (placeholder)
- `bridge_offer_skill`
- `graph_metrics`
- `G_compass_v0.gpickle`
- `transition_matrix_baseline`

✅ Visualisations :
- distributions par métiers, villes, types de contrat
- histogrammes de salaires (si dispo)

🔜 Prochaines étapes :
- Remplacer les placeholders par :
  - `dim_job` à partir de l'API ROME métiers
  - `dim_skill` à partir de l'API ROME compétences
  - bridges job-skill & job-activity à partir de ROME
- Enrichir le graphe Compass (job–job, job–activity, skill–skill)
- Définir une vraie matrice de transition pour Monte Carlo
  (similarité de compétences, proximité dans le graphe, historique des offres)

Ce notebook est prêt à être versionné et itéré avec Claude Code.